# 03 · SHAP — Global & Local Explanations

In [ ]:
!pip install shap lime matplotlib seaborn pandas numpy scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.datasets import load_breast_cancer
import shap
import lime
import lime.lime_tabular

np.random.seed(42)


In [ ]:
# Load the Breast Cancer Wisconsin dataset (used throughout this course)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Train the black-box model (Random Forest)

In [ ]:
# Train a Random Forest — a strong but harder-to-interpret "black box" model
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")


## SHAP: Global & Local Explanations

In [ ]:
# Initialize SHAP explainer
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# --- GLOBAL EXPLANATION: Feature importance summary ---
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values[:, :, 1], X_test, feature_names=data.feature_names, show=False)
plt.title("SHAP Global Feature Importance")
plt.tight_layout()
plt.show()

# --- LOCAL EXPLANATION: Single prediction ---
patient_idx = 0
shap.force_plot(explainer.expected_value[1], shap_values[patient_idx, :, 1],
                X_test.iloc[patient_idx,:], feature_names=data.feature_names,
                matplotlib=True, show=False)
plt.title(f"SHAP Explanation for Single Patient (Prediction: {y_pred_rf[patient_idx]})")
plt.tight_layout()
plt.show()

# --- DETAILED LOCAL EXPLANATION ---
shap_df = pd.DataFrame({
    'feature': data.feature_names,
    'shap_value': shap_values[patient_idx, :, 1]
}).sort_values('shap_value', key=abs, ascending=False)

print(f"\nPrediction for patient {patient_idx}: {'Benign' if y_pred_rf[patient_idx]==1 else 'Malignant'}")
print("Top features pushing towards Benign (positive SHAP) or Malignant (negative SHAP):")
print(shap_df.head(10))


## SHAP Dependence Plots

In [ ]:
# How does one feature affect predictions?
feature_of_interest = 'mean texture'
plt.figure(figsize=(10, 6))
shap.dependence_plot(feature_of_interest, shap_values[:, :, 1], X_test,
                     feature_names=data.feature_names, interaction_index='mean area',
                     show=False)
plt.title(f"SHAP Dependence: {feature_of_interest}")
plt.tight_layout()
plt.show()
